# Step 5: Evaluation

Evaluate the fine-tuned model against the test set.

**What this notebook covers:**
- Traditional metrics (BLEU, ROUGE, exact match)
- SQL execution accuracy (run queries and compare results)
- LLM-as-judge scoring
- Per-category breakdown (difficulty, SQL type)
- Comparison with GPT-4 baseline

# ⚠️ IMPORTANT - READ BEFORE RUNNING

**This notebook will RE-EVALUATE and OVERWRITE existing results.**

## Purpose
This is an **educational walkthrough** demonstrating how the evaluation pipeline works. It will:
- Re-run all evaluation metrics from scratch
- OVERWRITE any existing evaluation results

## When to Use This Notebook
✅ **LEARNING**: Understanding how evaluation works  
✅ **DEVELOPMENT**: Testing new evaluation metrics  
✅ **FRESH START**: Starting completely from scratch

## When NOT to Use This Notebook
❌ **COMPARISON**: You want to compare teacher vs student models with cost analysis

## What You Should Run Instead
If you have completed training and want a comprehensive teacher vs student comparison, run:
- `notebooks/07_comparison_glm.ipynb` (if you have GLM API)
- `notebooks/07_comparison_anthropic.ipynb` (if you have Anthropic API)

These comparison notebooks include:
- Teacher model evaluation (Sonnet/GLM vs Llama)
- Quality metrics comparison
- Cost analysis and break-even calculations
- Visualizations and deployment recommendations

See [docs/notebook-guide.md](docs/notebook-guide.md) for complete guidance.

---

# Step 5: Evaluation

Evaluate the fine-tuned model against the test set.

**What this notebook covers:**
- Traditional metrics (BLEU, ROUGE, exact match)
- SQL execution accuracy (run queries and compare results)
- LLM-as-judge scoring
- Per-category breakdown (difficulty, SQL type)
- Comparison with GPT-4 baseline

In [ ]:
import sys
sys.path.insert(0, '..')

import json
from src.evaluate.metrics import evaluate_batch
from src.evaluate.judge import LLMJudge
from src.evaluate.benchmark import BenchmarkRunner

# Try importing unsloth for model loading (may not be available on all systems)
try:
    from unsloth import FastLanguageModel
    UNSLOTH_AVAILABLE = True
except ImportError:
    UNSLOTH_AVAILABLE = False
    print("Warning: Unsloth not available. Model loading will fail.")

In [ ]:
# Load test data
with open('data/curated/test.jsonl') as f:
    test_data = [json.loads(line) for line in f]
print(f'Test set: {len(test_data)} examples')

In [ ]:
# Load fine-tuned model
if not UNSLOTH_AVAILABLE:
    raise ImportError("Unsloth is required to load the model. Install with: pip install unsloth")
model, tokenizer = FastLanguageModel.from_pretrained('models/sql-llama-8b-lora')

In [ ]:
# Run benchmark
runner = BenchmarkRunner()
results = runner.run(
    model=model,
    tokenizer=tokenizer,
    test_data=test_data,
    metrics=['exact_match', 'bleu', 'rouge', 'execution_accuracy']
)
runner.report(results)

In [ ]:
# LLM-as-judge evaluation (on a sample)
judge = LLMJudge()
judge_results = judge.judge_batch(test_data[:50])
avg_score = sum(r['score'] for r in judge_results) / len(judge_results)
print(f'Average judge score: {avg_score:.2f} / 5.0')

In [ ]:
# Per-difficulty breakdown
from collections import defaultdict
by_difficulty = defaultdict(list)
for r in results['per_example']:
    by_difficulty[r['difficulty']].append(r['execution_accuracy'])

for diff, scores in sorted(by_difficulty.items()):
    print(f'{diff}: {sum(scores)/len(scores):.1%} execution accuracy ({len(scores)} examples)')